# XLNet: Mathematical Formulation and Key Concepts

XLNet combines autoregressive models with permutation language modeling. Its key mathematical aspects include:

## 1. Permutation Language Modeling

- Considers all possible permutations of factorization order
- For sequence $x = [x_1, x_2, \ldots, x_T]$, maximizes likelihood over all permutations
- Factorization order: $P(x) = \prod_{t=1}^T P(x_{\pi_t} | x_{\pi_{<t}})$
  where $\pi \in Z_T$ (set of all permutations)

## 2. Two-Stream Self-Attention

- Content stream: Processes actual content
- Query stream: Handles positional information
- Hidden states: $h_i$ (content), $g_i$ (query)
- Updates:
  - Content: $h_i^{(l+1)} = \text{SelfAttention}(h_{\leq i}^{(l)}) + \text{FFN}(h_i^{(l)})$
  - Query: $g_i^{(l+1)} = \text{SelfAttention}(g_{\leq i-1}^{(l)}, h_{\leq i}^{(l)}) + \text{FFN}(g_i^{(l)})$

## 3. Objective Function

- Maximizes log-likelihood over all permutations:
  $\mathcal{L}_{\text{XLNet}} = \mathbb{E}_{\pi \sim Z_T} \left[ \sum_{t=1}^T \log P(x_{\pi_t} | x_{\pi_{<t}}) \right]$

## 4. Transformer-XL Architecture

- Uses segment-level recurrence for long-range dependencies
- Recurrence mechanism: $h_i^{(l)} = \text{SelfAttention}(h_{i-n:i}^{(l-1)}, h_{i-n:i-1}^{(l)})$
  where $n$ is segment length

## 5. Positional Encodings

- Uses relative positional encodings
- Attention mechanism:
  $\text{Attention}(Q, K, V) = \text{softmax}\left( \frac{QK^T + QR^T + S}{\sqrt{d_k}} \right) V$
  where $Q$, $K$ are query/key matrices, $R$ is relative positional encoding, $S$ is learnable bias

This formulation allows XLNet to capture bidirectional contexts while maintaining autoregressive properties, leading to superior performance on various NLP tasks.

In [1]:
!pip install torch==2.0.1 numpy

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint
import math

class RelativePositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len
        self.embeddings = nn.Parameter(torch.Tensor(max_len * 2 + 1, d_model))
        nn.init.normal_(self.embeddings, mean=0, std=0.02)

    def forward(self, q_len, k_len):
        range_vec = torch.arange(k_len, device=self.embeddings.device)
        range_mat = range_vec.unsqueeze(0).expand(q_len, -1)
        distance_mat = range_mat - torch.arange(q_len, device=self.embeddings.device).unsqueeze(1)
        distance_mat_clipped = torch.clamp(distance_mat + self.max_len, 0, 2 * self.max_len)
        return self.embeddings[distance_mat_clipped]

class TwoStreamAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.o_linear = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        self.rel_pos_encoding = RelativePositionalEncoding(self.head_dim)

    def forward(self, content, query, mask, mems=None):
        batch_size, seq_len, _ = content.size()

        if mems is not None:
            content = torch.cat([mems, content], dim=1)
            if mask is not None:
                mask = torch.cat([torch.ones_like(mask[:, :, :, :mems.size(1)]), mask], dim=-1)

        q = self.q_linear(query).view(batch_size, -1, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_linear(content).view(batch_size, -1, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_linear(content).view(batch_size, -1, self.n_heads, self.head_dim).transpose(1, 2)

        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        rel_pos = self.rel_pos_encoding(seq_len, content.size(1))
        rel_pos_scores = torch.einsum('bhld,lrd->bhlr', q, rel_pos)
        attn_scores = attn_scores + rel_pos_scores

        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

        attn_probs = F.softmax(attn_scores, dim=-1)
        attn_probs = self.dropout(attn_probs)

        context = torch.matmul(attn_probs, v)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        output = self.o_linear(context)
        return output

class PermutationLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, mem_len=0):
        super().__init__()
        self.d_model = d_model
        self.mem_len = mem_len
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([TwoStreamAttention(d_model, n_heads) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x, permutation, mems=None, use_checkpoint=True):
        batch_size, seq_len = x.size()

        x = self.embedding(x)

        mask = self._create_mask(batch_size, seq_len, permutation)

        content_stream = x
        query_stream = torch.zeros_like(x)

        new_mems = []
        for i, layer in enumerate(self.layers):
            mem = mems[i] if mems is not None else None
            if mem is not None:
                mem = mem.repeat(batch_size, 1, 1)  # Repeat memory for each item in the batch
            if use_checkpoint:
                query_stream = checkpoint(layer, content_stream, query_stream, mask, mem)
                content_stream = checkpoint(layer, content_stream, content_stream, mask, mem)
            else:
                query_stream = layer(content_stream, query_stream, mask, mem)
                content_stream = layer(content_stream, content_stream, mask, mem)

            if self.mem_len > 0:
                new_mems.append(content_stream[:, -self.mem_len:].detach())

        output = self.norm(query_stream)
        return self.fc(output), new_mems

    def _create_mask(self, batch_size, seq_len, permutation):
        mask = torch.zeros(batch_size, 1, seq_len, seq_len, device=permutation.device)
        for b in range(batch_size):
            for i in range(seq_len):
                mask[b, 0, i, permutation[b, :i+1]] = 1
        return mask

    def init_mems(self):
        mems = []
        param = next(self.parameters())
        for _ in self.layers:
            empty_mem = torch.zeros(1, self.mem_len, self.d_model, dtype=param.dtype, device=param.device)
            mems.append(empty_mem)
        return mems

class SyntheticDataset(Dataset):
    def __init__(self, num_samples, seq_length, vocab_size):
        self.data = []
        for _ in range(num_samples):
            seq = torch.randint(1, vocab_size, (seq_length,))
            target = (seq + 1) % vocab_size  # Next number in sequence
            self.data.append((seq, target))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def permutation_language_modeling_loss(model, batch, targets, mems, device):
    batch_size, seq_len = batch.size()

    # Generate random permutations
    permutations = torch.stack([torch.randperm(seq_len) for _ in range(batch_size)]).to(device)

    # Create factorization order
    factorization_order = torch.argsort(permutations, dim=1)

    # Forward pass
    outputs, new_mems = model(batch, permutations, mems)

    # Compute loss only on the factorization positions
    loss = 0
    for i in range(seq_len):
        fact_pos = factorization_order[:, i]
        target_pos = (fact_pos + 1) % seq_len  # Next token in the sequence
        loss += F.cross_entropy(outputs[torch.arange(batch_size), fact_pos], targets[torch.arange(batch_size), target_pos])

    return loss / seq_len, new_mems

def train(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    mems = model.init_mems()
    for batch, targets in dataloader:
        batch, targets = batch.to(device), targets.to(device)
        optimizer.zero_grad()

        loss, new_mems = permutation_language_modeling_loss(model, batch, targets, mems, device)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        # Update mems
        mems = [m[:1].detach() for m in new_mems]  # Keep only one item from each memory

    return total_loss / len(dataloader)

def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    mems = model.init_mems()
    with torch.no_grad():
        for batch, targets in dataloader:
            batch, targets = batch.to(device), targets.to(device)
            loss, new_mems = permutation_language_modeling_loss(model, batch, targets, mems, device)
            total_loss += loss.item()
            mems = [m[:1].detach() for m in new_mems]  # Keep only one item from each memory
    return total_loss / len(dataloader)

def generate(model, input_seq, max_length):
    model.eval()
    device = input_seq.device
    generated = input_seq
    mems = model.init_mems()

    with torch.no_grad():
        for _ in range(max_length - input_seq.size(1)):
            permutation = torch.arange(generated.size(1), device=device).unsqueeze(0)
            output, new_mems = model(generated, permutation, mems)
            next_token = output[:, -1, :].argmax(dim=-1).unsqueeze(-1)
            generated = torch.cat([generated, next_token], dim=-1)
            mems = [m[:1].detach() for m in new_mems]  # Keep only one item from each memory

    return generated

# Hyperparameters
vocab_size = 100
d_model = 128
n_heads = 4
n_layers = 2
seq_len = 10
batch_size = 32
num_epochs = 10
learning_rate = 0.001
mem_len = 5

# Create synthetic dataset
train_dataset = SyntheticDataset(1000, seq_len, vocab_size)
val_dataset = SyntheticDataset(200, seq_len, vocab_size)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

# Initialize model and move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PermutationLM(vocab_size, d_model, n_heads, n_layers, mem_len).to(device)

# Loss and optimizer
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    train_loss = train(model, train_dataloader, optimizer, device)
    val_loss = evaluate(model, val_dataloader, device)
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

# Test generation
input_seq = torch.randint(0, vocab_size, (1, 5)).to(device)
generated = generate(model, input_seq, max_length=15)
print("Input sequence:", input_seq.squeeze().tolist())
print("Generated sequence:", generated.squeeze().tolist())

/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:31: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn("None of the inputs have requires_grad=True. Gradients will be None")


Epoch 1/10, Train Loss: 4.5284, Val Loss: 4.3210
Epoch 2/10, Train Loss: 3.9035, Val Loss: 3.2210
Epoch 3/10, Train Loss: 3.1035, Val Loss: 2.9180
Epoch 4/10, Train Loss: 2.8973, Val Loss: 2.7048
Epoch 5/10, Train Loss: 2.6693, Val Loss: 2.5418
Epoch 6/10, Train Loss: 2.5312, Val Loss: 2.5116
Epoch 7/10, Train Loss: 2.4879, Val Loss: 2.4940
Epoch 8/10, Train Loss: 2.4219, Val Loss: 2.4567
Epoch 9/10, Train Loss: 2.3700, Val Loss: 2.2926
Epoch 10/10, Train Loss: 2.3089, Val Loss: 2.2748
Input sequence: [59, 66, 35, 83, 48]
Generated sequence: [59, 66, 35, 83, 48, 60, 67, 67, 6, 6, 60, 6, 6, 6, 6]


This is not XLNet. While this implementation is inspired by some of XLNet's key concepts, it's a significantly simplified version that lacks many of XLNet's advanced features and optimizations. Here are some key differences:

1. Scale: XLNet is typically much larger, with hundreds of millions to billions of parameters, while this is a small-scale implementation.

2. Training data: XLNet is trained on massive amounts of real-world text data, while this uses a small synthetic dataset.

3. Complexity: This implementation is much simpler and lacks many of XLNet's advanced features.

4. Specific XLNet features: This model doesn't include some XLNet-specific elements like the segment recurrence mechanism used for long sequences, or the specialized initialization and training techniques.

5. Tokenization: XLNet uses SentencePiece tokenization, while this model uses simple integer tokens.

6. Pre-training objectives: XLNet uses more sophisticated pre-training objectives and techniques.

7. Optimization: XLNet employs various optimization techniques for efficient training of large models, which are not implemented here.

8. Fine-tuning: XLNet is designed to be fine-tuned on various downstream tasks, which isn't implemented in this script.

This implementation is more of an educational example that demonstrates some concepts inspired by XLNet, such as permutation language modeling and two-stream attention. It's a simplified model that shares some architectural similarities with XLNet, but it's not a full or accurate reproduction of XLNet itself.